# 06 KMeans and Random Forest

Creates IAQ clusters and trains Random Forest models for cluster-based analysis.

**Run order:** this notebook is part of the AirAware workflow and uses the shared repository folders: `data/`, `data/processed/`, and `outputs/`.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.inspection import permutation_importance
 
warnings.filterwarnings("ignore")

In [ ]:
# ============================================================
# Project paths
# ============================================================
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

DATA_DIR.mkdir(exist_ok=True)
PROCESSED_DATA_DIR.mkdir(exist_ok=True)
OUTPUTS_DIR.mkdir(exist_ok=True)


In [ ]:
DATA_PATH = DATA_DIR / "synthetic_air_quality_dataset_final.csv"
 
print("=" * 60)
print("STEP 1 — Loading data")
print("=" * 60)
 
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
print("\n" + "=" * 60)
print("STEP 2 — Preprocessing")
print("=" * 60)
 
# Drop rows with NaN in the three pollutant rate columns
POLLUTANTS = ["pm25_rate", "pm10_rate", "no2_rate"]
df = df.dropna(subset=POLLUTANTS)
print(f"Rows after dropping NaN in pollutant columns: {len(df)}")
 
# Encode categorical columns that will be used as RF features
CAT_COLS = [
    "time_of_day",
    "motion_type",
    "environment_type",
    "activity_type",
    "cooking_method",
    "appliance_type",
]
le = LabelEncoder()
for col in CAT_COLS:
    df[col + "_enc"] = le.fit_transform(df[col].astype(str))
 
# Boolean → int
df["cooking_event_int"] = df["cooking_event"].astype(int)
 
print("Categorical columns label-encoded.")
print(df[POLLUTANTS].describe())

In [ ]:
print("\n" + "=" * 60)
print("STEP 3 — Computing Indoor Pollution Index (IPI)")
print("=" * 60)
 
scaler_mm = MinMaxScaler()
pollutant_normalised = scaler_mm.fit_transform(df[POLLUTANTS])
norm_pm25, norm_pm10, norm_no2 = (
    pollutant_normalised[:, 0],
    pollutant_normalised[:, 1],
    pollutant_normalised[:, 2],
)
 
W_PM25, W_NO2, W_PM10 = 0.50, 0.30, 0.20
df["IPI"] = W_PM25 * norm_pm25 + W_NO2 * norm_no2 + W_PM10 * norm_pm10
 
print(f"IPI weights  →  PM2.5: {W_PM25}  |  NO2: {W_NO2}  |  PM10: {W_PM10}")
print(f"IPI range    →  min={df['IPI'].min():.4f}  max={df['IPI'].max():.4f}")
print(f"IPI mean     →  {df['IPI'].mean():.4f}  |  std={df['IPI'].std():.4f}")

In [ ]:
print("\n" + "=" * 60)
print("STEP 4 — KMeans Clustering on pollutant space")
print("=" * 60)
 
# Scale pollutants to zero mean / unit variance for KMeans
scaler_std = StandardScaler()
X_cluster = scaler_std.fit_transform(df[POLLUTANTS])
 
# Elbow method to choose K
inertia = []
K_RANGE = range(2, 11)
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertia.append(km.inertia_)
 
# Plot elbow curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(K_RANGE), inertia, marker="o", color="steelblue", linewidth=2)
ax.set_xlabel("Number of clusters (K)")
ax.set_ylabel("Inertia (within-cluster SSE)")
ax.set_title("KMeans Elbow Curve")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("outputs/kmeans_elbow.png", dpi=150)
plt.close()
print("Elbow plot saved → kmeans_elbow.png")
 
# Fit final KMeans with chosen K (visual elbow typically around 4)
K_FINAL = 4
km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df["cluster"] = km_final.fit_predict(X_cluster)
 
print(f"\nFinal KMeans  →  K = {K_FINAL}")
print("Cluster distribution:")
print(df["cluster"].value_counts().sort_index().to_string())
 
# Cluster profile — mean pollutant rates and IPI per cluster
cluster_profile = df.groupby("cluster")[POLLUTANTS + ["IPI"]].mean().round(3)
cluster_profile["size"] = df["cluster"].value_counts().sort_index()
print("\nCluster profiles (mean values):")
print(cluster_profile.to_string())
 
# Scatter plot: PM2.5 vs NO2 coloured by cluster
fig, ax = plt.subplots(figsize=(8, 5))
colours = cm.tab10(np.linspace(0, 0.4, K_FINAL))
for c in range(K_FINAL):
    mask = df["cluster"] == c
    ax.scatter(
        df.loc[mask, "pm25_rate"],
        df.loc[mask, "no2_rate"],
        s=5,
        alpha=0.4,
        color=colours[c],
        label=f"Cluster {c}",
    )
ax.set_xlabel("PM2.5 rate (mg/min)")
ax.set_ylabel("NO2 rate (mg/min)")
ax.set_title("KMeans Clusters — PM2.5 vs NO2")
ax.legend(markerscale=3)
plt.tight_layout()
plt.savefig("outputs/kmeans_clusters.png", dpi=150)
plt.close()
print("Cluster scatter plot saved → kmeans_clusters.png")

In [ ]:
print("\n" + "=" * 60)
print("STEP 5 — Random Forest Regressor to predict IPI")
print("=" * 60)
 
# Feature set: cluster label + contextual variables (NO raw pollutant rates)
# We deliberately exclude pm25_rate, pm10_rate, no2_rate from RF features
# because they are the inputs that define IPI — using them would be data leakage.
FEATURE_COLS = [
    "cluster",
    "cooking_event_int",
    "cooking_duration_min",
    "gas_cooking_or_gas_hob",
    "wood_burning_event",
    "smoking_vaping_exposure",
    "window_open",
    "extractor_fan_on",
    "outdoor_pm25",
    "outdoor_pm10",
    "outdoor_no2",
    "outdoor_temp",
    "duration_min",
] + [c + "_enc" for c in CAT_COLS]
 
X = df[FEATURE_COLS].fillna(0)
y = df["IPI"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {len(X_train)}  |  Test size: {len(X_test)}")
 
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)
print("Random Forest trained.")
 
# ─────────────────────────────────────────────
# 6. EVALUATION
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6 — Evaluation")
print("=" * 60)
 
y_pred_train = rf.predict(X_train)
y_pred_test = rf.predict(X_test)
 
for split, y_true, y_pred in [
    ("Train", y_train, y_pred_train),
    ("Test ", y_test, y_pred_test),
]:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{split}  →  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}")
 
# Predicted vs Actual scatter
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred_test, s=3, alpha=0.3, color="steelblue")
lims = [0, 1]
ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlabel("Actual IPI")
ax.set_ylabel("Predicted IPI")
ax.set_title("Random Forest — Predicted vs Actual IPI")
ax.legend()
plt.tight_layout()
plt.savefig("outputs/rf_predicted_vs_actual.png", dpi=150)
plt.close()
print("Predicted vs Actual plot saved → rf_predicted_vs_actual.png")

In [ ]:
# ─────────────────────────────────────────────
# 7. FEATURE IMPORTANCE
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 7 — Feature Importance")
print("=" * 60)
 
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS)
importances_sorted = importances.sort_values(ascending=False)
print(importances_sorted.round(4).to_string())
 
# Plot top 15 features
top_n = min(15, len(importances_sorted))
fig, ax = plt.subplots(figsize=(8, 5))
importances_sorted.head(top_n).sort_values().plot(
    kind="barh", color="steelblue", ax=ax
)
ax.set_xlabel("Feature Importance (Gini impurity)")
ax.set_title(f"Top {top_n} Features — Random Forest")
plt.tight_layout()
plt.savefig("outputs/rf_feature_importance.png", dpi=150)
plt.close()
print(f"Feature importance plot saved → rf_feature_importance.png")

Since IPI and cluster will have high corlinearility since cluster varaible are derived from pollutant only. This means the model is largely learning a circular relationship rather than genuinely predicting indoor air quality from contextual behaviour. So let me try out without any clustering

### Direct Randomforest regressor without kmeans

In [ ]:
print("\n" + "=" * 60)
print("STEP 5 — Random Forest Regressor to predict IPI Without Cluster Variable")
print("=" * 60)
 
# Feature set: cluster label + contextual variables (NO raw pollutant rates)
# We deliberately exclude pm25_rate, pm10_rate, no2_rate from RF features
# because they are the inputs that define IPI — using them would be data leakage.
FEATURE_COLS = [
    "cooking_event_int",
    "cooking_duration_min",
    "gas_cooking_or_gas_hob",
    "wood_burning_event",
    "smoking_vaping_exposure",
    "window_open",
    "extractor_fan_on",
    "outdoor_pm25",
    "outdoor_pm10",
    "outdoor_no2",
    "outdoor_temp",
    "duration_min",
] + [c + "_enc" for c in CAT_COLS]
X = df[FEATURE_COLS].fillna(0)
y = df["IPI"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {len(X_train)}  |  Test size: {len(X_test)}")
 
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)
print("Random Forest trained.")
 
# ─────────────────────────────────────────────
# 6. EVALUATION
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6 — Evaluation")
print("=" * 60)
 
y_pred_train = rf.predict(X_train)
y_pred_test = rf.predict(X_test)
 
for split, y_true, y_pred in [
    ("Train", y_train, y_pred_train),
    ("Test ", y_test, y_pred_test),
]:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{split}  →  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}")
 
# Predicted vs Actual scatter
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred_test, s=3, alpha=0.3, color="steelblue")
lims = [0, 1]
ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlabel("Actual IPI")
ax.set_ylabel("Predicted IPI")
ax.set_title("Random Forest — Predicted vs Actual IPI")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS)
importances_sorted = importances.sort_values(ascending=False)
print(importances_sorted.round(4).to_string())

From above result I infer that we can stil use kmeans on other variable, let's check the model performance from it

## try kmenas without pollutant

In [ ]:
df.head(1)

In [ ]:
print("\n" + "=" * 60)
print("KMeans Clustering without  pollutant space")
print("=" * 60)
 
# Scale pollutants to zero mean / unit variance for KMeans
feature_1 = ["duration_min", "outdoor_pm25", "outdoor_pm10", "outdoor_no2","gas_cooking_or_gas_hob", "cooking_duration_min",
"wood_burning_event", "smoking_vaping_exposure", "window_open", "extractor_fan_on"] + [c + "_enc" for c in CAT_COLS]
scaler_std = StandardScaler()

X_cluster = scaler_std.fit_transform(df[feature_1])
 
# Elbow method to choose K
inertia = []
K_RANGE = range(2, 11)
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_cluster)
    inertia.append(km.inertia_)
 
# Plot elbow curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(K_RANGE), inertia, marker="o", color="steelblue", linewidth=2)
ax.set_xlabel("Number of clusters (K)")
ax.set_ylabel("Inertia (within-cluster SSE)")
ax.set_title("KMeans Elbow Curve")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Fit final KMeans with chosen K (visual elbow typically around 4)
K_FINAL = 4
km_final = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df["cluster"] = km_final.fit_predict(X_cluster)
 
print(f"\nFinal KMeans  →  K = {K_FINAL}")
print("Cluster distribution:")
print(df["cluster"].value_counts().sort_index().to_string())

In [ ]:
pollutant_profile = df.groupby("cluster")[POLLUTANTS + ["IPI"]].mean().round(3)
pollutant_profile["count"] = df["cluster"].value_counts().sort_index()
 
# Rank IPI to assign pollution level label
pollutant_profile["IPI_rank"] = pollutant_profile["IPI"].rank().astype(int)
level_map = {1: "LOW", 2: "MODERATE-LOW", 3: "MODERATE-HIGH", 4: "HIGH"}
pollutant_profile["pollution_level"] = pollutant_profile["IPI_rank"].map(level_map)
 
print(pollutant_profile.to_string())

In [ ]:
behav_summary_cols = [
    "cooking_event_int",
    "gas_cooking_or_gas_hob",
    "cooking_duration_min",
    "window_open",
    "extractor_fan_on",
    "wood_burning_event",
    "smoking_vaping_exposure",
]
behav_profile = df.groupby("cluster")[behav_summary_cols].mean().round(3)
 
# Most common cooking method per cluster
behav_profile["dominant_cooking_method"] = (
    df.groupby("cluster")["cooking_method"]
    .agg(lambda x: x.value_counts().index[0])
)
behav_profile["dominant_appliance"] = (
    df.groupby("cluster")["appliance_type"]
    .agg(lambda x: x.value_counts().index[0])
)
 
print(behav_profile.to_string())

In [ ]:
summary = pd.concat([pollutant_profile[["pollution_level", "IPI", "count"]],
                     behav_profile[["cooking_event_int", "gas_cooking_or_gas_hob",
                                    "window_open", "extractor_fan_on",
                                    "dominant_cooking_method", "dominant_appliance"]]], axis=1)
print(summary.to_string())

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
colours = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c"]  # green→red for LOW→HIGH
 
# Sort clusters by IPI level for colour assignment
cluster_order = pollutant_profile["IPI"].sort_values().index.tolist()
 
for i, c in enumerate(cluster_order):
    ax = axes[i]
    data = df.loc[df["cluster"] == c, "IPI"]
    level = pollutant_profile.loc[c, "pollution_level"]
    ax.hist(data, bins=50, color=colours[i], edgecolor="white", alpha=0.85)
    ax.set_title(f"Cluster {c} — {level}\nMean IPI={data.mean():.3f}  n={len(data):,}")
    ax.set_xlabel("IPI")
    ax.set_ylabel("Count")
    ax.grid(True, alpha=0.2)
 
plt.suptitle("IPI Distribution per Behavioural Cluster", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
print("\n" + "=" * 60)
print("STEP 5 — Random Forest Regressor to predict IPI Without Cluster Variable")
print("=" * 60)
 
# Feature set: cluster label + contextual variables (NO raw pollutant rates)
# We deliberately exclude pm25_rate, pm10_rate, no2_rate from RF features
# because they are the inputs that define IPI — using them would be data leakage.
FEATURE_COLS = [
    "cluster",
    "cooking_event_int",
    "cooking_duration_min",
    "gas_cooking_or_gas_hob",
    "wood_burning_event",
    "smoking_vaping_exposure",
    "window_open",
    "extractor_fan_on",
    "outdoor_pm25",
    "outdoor_pm10",
    "outdoor_no2",
    "outdoor_temp",
    "duration_min",
] + [c + "_enc" for c in CAT_COLS]
X = df[FEATURE_COLS].fillna(0)
y = df["IPI"]
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train size: {len(X_train)}  |  Test size: {len(X_test)}")
 
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)
rf.fit(X_train, y_train)
print("Random Forest trained.")
 
# ─────────────────────────────────────────────
# 6. EVALUATION
# ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("STEP 6 — Evaluation")
print("=" * 60)
 
y_pred_train = rf.predict(X_train)
y_pred_test = rf.predict(X_test)
 
for split, y_true, y_pred in [
    ("Train", y_train, y_pred_train),
    ("Test ", y_test, y_pred_test),
]:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"{split}  →  MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}")
 
# Predicted vs Actual scatter
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred_test, s=3, alpha=0.3, color="steelblue")
lims = [0, 1]
ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")
ax.set_xlabel("Actual IPI")
ax.set_ylabel("Predicted IPI")
ax.set_title("Random Forest — Predicted vs Actual IPI")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS)
importances_sorted = importances.sort_values(ascending=False)
print(importances_sorted.round(4).to_string())